In [1]:
# 检查cdr是否在序列中唯一，对所有任务都通用
def find_all_overlapping(text, substring):
    indices = []
    start_index = 0

    while True:
        index = text.find(substring, start_index)
        if index == -1:
            break
        indices.append(index)
        start_index = index + 1 
        
    return indices

### 1. science 4个 CR

In [23]:
import pandas as pd
import os
from abnumber import Chain

cr6261_h1_heavy_seq = "EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK"
cr6261_h9_heavy_seq = "EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK"
cr9114_h1_heavy_seq = "QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH"
cr9114_h3_heavy_seq = "QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH"

sequences = {
    'cr6261_h1': cr6261_h1_heavy_seq,
    'cr6261_h9': cr6261_h9_heavy_seq,
    'cr9114_h1': cr9114_h1_heavy_seq,
    'cr9114_h3': cr9114_h3_heavy_seq
}

In [24]:
pdb_id_list = []
seq_list = []
chain_id_list = []
partner_list = []
cdr1_list = []
cdr2_list = []
cdr3_list = []

chain_id = 'A'
partner = 'AB_CD'

for pdb_id, seq in sequences.items():
    chain = Chain(seq, scheme='chothia')
    cdr1 = chain.cdr1_seq
    cdr2 = chain.cdr2_seq
    cdr3 = chain.cdr3_seq

    # 检查cdr在序列中是否唯一
    cdr1_indices = find_all_overlapping(seq, cdr1)
    cdr2_indices = find_all_overlapping(seq, cdr2)
    cdr3_indices = find_all_overlapping(seq, cdr3)
    assert len(cdr1_indices) == 1, f"{pdb_id} CDR1 not unique"
    assert len(cdr2_indices) == 1, f"{pdb_id} CDR2 not unique"
    assert len(cdr3_indices) == 1, f"{pdb_id} CDR3 not unique"

    pdb_id_list.append(pdb_id)
    seq_list.append(seq)
    chain_id_list.append(chain_id)
    partner_list.append(partner)
    cdr1_list.append(cdr1)
    cdr2_list.append(cdr2)
    cdr3_list.append(cdr3)

data = {
    'pdb_id': pdb_id_list,
    'antibody_chain': chain_id_list,
    'partner': partner_list,
    'sequence': seq_list,
    'cdr1': cdr1_list,
    'cdr2': cdr2_list,
    'cdr3': cdr3_list
}

df = pd.DataFrame(data)
df.to_csv('/home/lfj/projects_dir/MERF/data/CR/cr_evo.csv', index=False)

### 2. SAbDab

In [2]:
# 可能需要从头处理数据。不用完全从头，借助一下从chothia中处理链以及序列范围的代码即可，参考自MFDesign summary.py
import json
import os
import yaml
import pandas as pd
import logging
from abnumber import Chain
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select, Selection

In [ ]:
# 对json_dir下的每个json文件进行处理，检查文件名所对应H chain是否严格为标号0（其实可以不用处理此步，原论文处理过）

json_dir = "/home/lfj/projects_dir/MERF/baselines/MFDesign/data/antibody_data/records/"

for filename in os.listdir(json_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(json_dir, filename)
        with open(file_path, 'r') as f:
            record = json.load(f)
        assert record['structure']['H_chain_id'] == 0, f"Chain ID mismatch in {filename}"
        assert record['structure']['L_chain_id'] == 1 or record['structure']['L_chain_id'] == None, f"Chain ID mismatch in {filename}"
        

In [3]:
# generate ours csv

yaml_dir = "/home/lfj/projects_dir/MERF/baselines/MFDesign/data/test_yaml_dir/ab/"
pdb_dir = "/home/lfj/projects_dir/MERF/baselines/MFDesign/data/raw_data/chothia/"
pdb_save_dir = "/home/lfj/projects_dir/MERF/data/sabdab/PDBs/"
data_path = "/home/lfj/projects_dir/MERF/baselines/MFDesign/MFDesign_TestSet_204_samples.csv"
data_df = pd.read_csv(data_path)

HEAVY_CHAIN_MAX_LEN = 113
LIGHT_CHAIN_MAX_LEN = 106

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        aa_sequence = []

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                if resname != 'HOH':
                    logging.warning(f'PDB {pdb_id} Chain {chain.get_id()} Res {resname} not AA')
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]
            aa_sequence.append(token)

        data[chain.get_id()] = ''.join(aa_sequence)

    return data

class ChainSelector(Select):
    def __init__(self, heavy_chain=None, light_chain=None, antigen_chains=None):
        self.heavy_chain = heavy_chain
        self.light_chain = light_chain
        self.antigen_chains = antigen_chains or []

    def accept_chain(self, chain):
        chain_id = chain.get_id()
        return (chain_id == self.heavy_chain or chain_id == self.light_chain or
                chain_id in self.antigen_chains)

    def accept_residue(self, residue):
        chain_id = residue.get_parent().get_id()
        res_id = residue.get_id()[1]

        if chain_id == self.heavy_chain:
            return res_id <= HEAVY_CHAIN_MAX_LEN
        elif chain_id == self.light_chain:
            return res_id <= LIGHT_CHAIN_MAX_LEN
        elif chain_id in self.antigen_chains:
            return True
        return False


In [5]:
# formal process

pdb_id_list = []
chain_id_list = []
partner_list = []
seq_list = []
cdr1_list = []
cdr2_list = []
cdr3_list = []

for idx in tqdm(range(len(data_df))):
    pdb_id_full = data_df.loc[idx, 'Sample_ID']
    pdb_id = data_df.loc[idx, 'PDB_ID']
    heavy_chain = data_df.loc[idx, 'Heavy_Chain']
    light_chain = data_df.loc[idx, 'Light_Chain']
    antigen_chain_str = data_df.loc[idx, 'Antigen_Chain']

    # print(light_chain)

    if not pd.notna(light_chain):
        light_chain = ''

    partner = f"{data_df.loc[idx, 'Heavy_Chain']}{light_chain}_{data_df.loc[idx, 'Antigen_Chain']}"
    antigen_chains = list(str(antigen_chain_str)) if pd.notna(antigen_chain_str) else []

    # 读取yaml文件
    yaml_file = os.path.join(yaml_dir, f"{pdb_id_full}.yaml")
    with open(yaml_file, 'r') as f:
        yaml_data = yaml.safe_load(f)
    
    # 获取截断的序列以及进行cdr标注
    for protein in yaml_data['sequences']:
        if protein['protein']['id'] == heavy_chain:
            seq_truncked = protein['protein']['ground_truth']
            seq_mask = protein['protein']['spec_mask']
            break

    chain = Chain(seq_truncked, scheme='chothia')
    cdr1 = chain.cdr1_seq
    cdr2 = chain.cdr2_seq
    cdr3 = chain.cdr3_seq

    # 检查cdr在序列中是否唯一
    cdr1_indices = find_all_overlapping(seq_truncked, cdr1)
    cdr2_indices = find_all_overlapping(seq_truncked, cdr2)
    cdr3_indices = find_all_overlapping(seq_truncked, cdr3)
    assert len(cdr1_indices) == 1, f"{pdb_id_full} CDR1 not unique"
    assert len(cdr2_indices) == 1, f"{pdb_id_full} CDR2 not unique"
    assert len(cdr3_indices) == 1, f"{pdb_id_full} CDR3 not unique"

    # 检查标注的cdr，是否和通过chothia标注已存储的mask一致
    all_cdr_residues_in_mask = [seq_truncked[i] for i, char in enumerate(seq_mask) if char == '1']
    if not cdr1 + cdr2 + cdr3 == ''.join(all_cdr_residues_in_mask):
        if cdr1[1:] + cdr2 + cdr3 == ''.join(all_cdr_residues_in_mask):
            cdr1 = cdr1[1:]  # 去掉第一个残基，这样的确和mask一致，那就统一这样
            print(f"CDR sequences do not match mask in {pdb_id_full}")
            print(f"CDRs: {cdr1} {cdr2} {cdr3}")
            print(f"Mask residues: {''.join(all_cdr_residues_in_mask)}") 
    
    # get the true sequence from pdb file
    pdb_file = os.path.join(pdb_dir, f'{pdb_id}.pdb')
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_file)
    pdb_structure = structure[0]

    # heavy_entity = pdb_structure[heavy_chain]
    # heavy_seqs = parse_biopython_structure(pdb_id, heavy_entity)
    # extracted_heavy_seq = heavy_seqs.get(heavy_chain, '')

    # if not seq_truncked == extracted_heavy_seq:
    #     print(f"Sequence mismatch for {pdb_id_full}:")
    #     print(f"YAML sequence:     {seq_truncked}")
    #     print(f"PDB extracted seq: {extracted_heavy_seq}")

    # save new pdb with only the antibody chain and antigen chain
    light_chain_arg = light_chain if (pd.notna(light_chain) and light_chain != '') else None
    selector = ChainSelector(
        heavy_chain=heavy_chain,
        light_chain=light_chain_arg,
        antigen_chains=antigen_chains
    )
    io = PDBIO()
    io.set_structure(structure)
    output_pdb_file = os.path.join(pdb_save_dir, f"{pdb_id_full}.pdb")
    io.save(output_pdb_file, select=selector)
    
    pdb_id_list.append(pdb_id_full)
    chain_id_list.append(heavy_chain)
    partner_list.append(partner)
    seq_list.append(seq_truncked)
    cdr1_list.append(cdr1)
    cdr2_list.append(cdr2)
    cdr3_list.append(cdr3)

data = {
    'pdb_id': pdb_id_list,
    'antibody_chain': chain_id_list,
    'partner': partner_list,
    'sequence': seq_list,
    'cdr1': cdr1_list,
    'cdr2': cdr2_list,
    'cdr3': cdr3_list
}

df = pd.DataFrame(data)
df.to_csv('/home/lfj/projects_dir/MERF/data/sabdab/sabdab_evo.csv', index=False)

  0%|          | 0/204 [00:00<?, ?it/s]

100%|██████████| 204/204 [00:59<00:00,  3.41it/s]


In [ ]:
# （legacy）检查原始npz中是否存储结构为非截断结构，这里根据处理自己csv时，发现的有截断的案例来检查（TMD，发现存储的是截断结构，无敌了）
npz_dir = "/home/lfj/projects_dir/MERF/baselines/MFDesign/data/antibody_data/structures/"
npz_file_list = [f for f in os.listdir(npz_dir) if f.endswith('.npz')]

import numpy as np
npz_path = os.path.join(npz_dir, '8wfm_A_a_B.npz')
# load processed npz file
data = np.load(npz_path)
print(data.files)

print(len("VQLVESGAEVKKPGSSVKVSCKAPFSSYAISWVRQAPGQGLEWMGGIIPIFGTPNYAQKFQGRVTITADESTSTAYMELSSLRSEDTAVYYCARGYCGGDCYSLELVWYFDLWGRGTLVTVS"))  # H链截断的

data['chains']

['atoms', 'bonds', 'residues', 'chains', 'connections', 'interfaces', 'mask']
122


array([('A', 0, 4, 0, 0,    0,  846,   0, 122),
       ('a', 0, 5, 0, 1,  846,  711, 122, 108),
       ('B', 0, 0, 0, 2, 1557, 1484, 230, 183)],
      dtype=[('name', '<U5'), ('mol_type', 'i1'), ('entity_id', '<i4'), ('sym_id', '<i4'), ('asym_id', '<i4'), ('atom_idx', '<i4'), ('atom_num', '<i4'), ('res_idx', '<i4'), ('res_num', '<i4')])

In [ ]:
# 检查自己存储的pdb的序列是否与npz中截断的长度都一致
import numpy as np
import pandas as pd
from tqdm import tqdm

npz_dir = "/home/lfj/projects_dir/MERF/baselines/MFDesign/data/antibody_data/structures/"

data_df = pd.read_csv(data_path)

for idx in tqdm(range(len(data_df))):
    pdb_id_full = data_df.loc[idx, 'Sample_ID']
    pdb_id = data_df.loc[idx, 'PDB_ID']
    heavy_chain = data_df.loc[idx, 'Heavy_Chain']
    light_chain = data_df.loc[idx, 'Light_Chain']
    antigen_chain_str = data_df.loc[idx, 'Antigen_Chain']

    if not pd.notna(light_chain):
        light_chain = ''

    antigen_chains = list(str(antigen_chain_str)) if pd.notna(antigen_chain_str) else []

    pdb_file = os.path.join(pdb_save_dir, f'{pdb_id_full}.pdb')
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id_full, pdb_file)
    pdb_structure = structure[0]

    npz_file = os.path.join(npz_dir, f"{pdb_id_full}.npz")
    data = np.load(npz_file)
    chains_in_npz = data['chains']  # dict
    npz_chain_seqlen_mapping = {item[0]: item[-1] for item in chains_in_npz}

    pdb_chain_seqlen_mapping = {}
    if heavy_chain == 'F':
        print(1)
    heavy_entity = pdb_structure[heavy_chain]
    heavy_seqs = parse_biopython_structure(pdb_id_full, heavy_entity)
    extracted_heavy_seq = heavy_seqs.get(heavy_chain, '')
    pdb_chain_seqlen_mapping[heavy_chain] = len(extracted_heavy_seq)

    if light_chain != '':
        light_entity = pdb_structure[light_chain]
        light_seqs = parse_biopython_structure(pdb_id_full, light_entity)
        extracted_light_seq = light_seqs.get(light_chain, '')
        pdb_chain_seqlen_mapping[light_chain] = len(extracted_light_seq)
    
    for antigen_chain in antigen_chains:
        antigen_entity = pdb_structure[antigen_chain]
        antigen_seqs = parse_biopython_structure(pdb_id_full, antigen_entity)
        extracted_antigen_seq = antigen_seqs.get(antigen_chain, '')
        pdb_chain_seqlen_mapping[antigen_chain] = len(extracted_antigen_seq)

    if pdb_chain_seqlen_mapping != npz_chain_seqlen_mapping:
        print(f"Sequence length mismatch in {pdb_id_full}")
        print(f"PDB lengths: {pdb_chain_seqlen_mapping}")
        print(f"NPZ lengths: {npz_chain_seqlen_mapping}")
        

 15%|█▍        | 30/204 [00:03<00:14, 11.83it/s]

1


 19%|█▊        | 38/204 [00:04<00:18,  9.13it/s]

1


 42%|████▏     | 86/204 [00:06<00:06, 17.56it/s]WARNING:root:PDB 9bu8_C_D_AB Chain A Res NAG not AA


...

 65%|██████▌   | 133/204 [00:09<00:03, 18.83it/s]WARNING:root:PDB 8q7o_A__C Chain C Res NAG not AA


1


 71%|███████   | 144/204 [00:10<00:03, 15.09it/s]WARNING:root:PDB 8s5w_B__A Chain A Res 2TA not AA


Sequence length mismatch in 8rh2_H_L_B
PDB lengths: {'H': 119, 'L': 104, 'B': 583}
NPZ lengths: {'H': 119, 'L': 103, 'B': 583}


100%|██████████| 204/204 [00:13<00:00, 14.75it/s]


In [ ]:
# 检查抗体的起始编号是否为1，如果不是则重编号

import os
import pandas as pd
import logging
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select, Selection
data_path = '/home/lfj/projects_dir/MERF/data/sabdab/sabdab_evo.csv'
data_df = pd.read_csv(data_path)

pdb_dir = '/home/lfj/projects_dir/MERF/data/sabdab/PDBs/'

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        
        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'pdb_id']
    
    h_chain = pdb_id.split('_')[1]
    l_chain = pdb_id.split('_')[2]

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    pdb_old_path = os.path.join(pdb_dir, f"{pdb_id}_old.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_old_path)
    pdb_structure = structure[0]

    h_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[h_chain])
    if list(h_chain_mapping.keys())[0] != 1:
        
        h_chain_model = pdb_structure[h_chain]

        chain_res_id_list = []
        # ========== 第一阶段：记录原始res id，且使用临时ID避免冲突 ==========
        residues_in_chain = list(h_chain_model.get_residues())
        for new_idx, residue in enumerate(residues_in_chain, start=1):
            old_id = residue.get_id()
            het_flag, old_seq_id, old_icode = old_id
            chain_res_id_list.append(new_idx)
            temp_id = (het_flag, 10000 + old_seq_id, old_icode)
            residue.id = temp_id
        
        # ========== 第二阶段：修改为最终编号 ==========
        for i, residue in enumerate(residues_in_chain):
            het_flag, temp_seq_id, old_icode = residue.get_id()
            new_seq_id = chain_res_id_list[i]
            new_id = (het_flag, new_seq_id, old_icode)
            residue.id = new_id
            
    if not l_chain == '':
        l_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[l_chain])
        if list(l_chain_mapping.keys())[0] != 1:
            
            l_chain_model = pdb_structure[l_chain]

            chain_res_id_list = []
            # ========== 第一阶段：记录原始res id，且使用临时ID避免冲突 ==========
            residues_in_chain = list(l_chain_model.get_residues())
            for new_idx, residue in enumerate(residues_in_chain, start=1):
                old_id = residue.get_id()
                het_flag, old_seq_id, old_icode = old_id
                chain_res_id_list.append(new_idx)
                temp_id = (het_flag, 10000 + old_seq_id, old_icode)
                residue.id = temp_id
            
            # ========== 第二阶段：修改为最终编号 ==========
            for i, residue in enumerate(residues_in_chain):
                het_flag, temp_seq_id, old_icode = residue.get_id()
                new_seq_id = chain_res_id_list[i]
                new_id = (het_flag, new_seq_id, old_icode)
                residue.id = new_id

    # move old pdb file to backup，跑完一次就不用了，而是读入时直接考虑old
    # pdb_old_path = pdb_path.replace('.pdb', '_old.pdb')
    # os.rename(pdb_path, pdb_old_path)

    io = PDBIO()
    io.set_structure(structure)
    io.save(pdb_path)
    print(f"Renumbered antibody chains in {pdb_id} and saved to {pdb_path}")

  1%|▏         | 3/204 [00:00<00:22,  9.02it/s]

Renumbered antibody chains in 8vuj_H_L_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8vuj_H_L_A.pdb
Renumbered antibody chains in 8r4d_B__A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8r4d_B__A.pdb
Renumbered antibody chains in 9bt8_A__B and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/9bt8_A__B.pdb


  2%|▏         | 4/204 [00:00<00:27,  7.17it/s]

Renumbered antibody chains in 8zes_E__D and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8zes_E__D.pdb
Renumbered antibody chains in 8u31_C_B_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8u31_C_B_A.pdb


  4%|▍         | 8/204 [00:00<00:22,  8.53it/s]

...

Renumbered antibody chains in 8rek_H_L_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8rek_H_L_A.pdb
Renumbered antibody chains in 8wo4_B_A_F and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8wo4_B_A_F.pdb


 99%|█████████▊| 201/204 [00:27<00:00,  8.78it/s]

Renumbered antibody chains in 8sic_A_B_E and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8sic_A_B_E.pdb
Renumbered antibody chains in 8ts0_H_L_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8ts0_H_L_A.pdb
Renumbered antibody chains in 9e6k_H_L_C and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/9e6k_H_L_C.pdb


100%|██████████| 204/204 [00:27<00:00,  7.29it/s]

Renumbered antibody chains in 8tl3_G_H_AB and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8tl3_G_H_AB.pdb
Renumbered antibody chains in 8v4f_B_D_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8v4f_B_D_A.pdb
Renumbered antibody chains in 8vul_H_L_A and saved to /home/lfj/projects_dir/MERF/data/sabdab/PDBs/8vul_H_L_A.pdb


### 3. 自己找的SARS-CoV-2

In [1]:
import json
import os
import yaml
import pandas as pd
import logging
from abnumber import Chain
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select, Selection
import requests
import abnumber


STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        aa_sequence = []

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                if resname != 'HOH':
                    logging.warning(f'PDB {pdb_id} Chain {chain.get_id()} Res {resname} not AA')
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]
            aa_sequence.append(token)

        data[chain.get_id()] = ''.join(aa_sequence)

    return data

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        
        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                # if resname != 'HOH':
                #     logging.warning(f'PDB {pdb_id} Chain {chain.get_id()} Res {resname} not AA')
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping


#### 3.1 分辨率 v1->v2

In [13]:
# 获取分辨率数据

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v1.csv"
data_df = pd.read_csv(data_path)

def get_resolution(pdb_id):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        try:
            resolution = data['rcsb_entry_info']['resolution_combined'][0]
            print(f"Resolution for PDB ID {pdb_id}: {resolution} Å")
            return resolution
        except (KeyError, IndexError):
            print(f"Resolution information not found for PDB ID {pdb_id}.")
            return 100
    else:
        print(f"Failed to retrieve data for PDB ID {pdb_id}. Status code: {response.status_code}")
        return 100

resolution_list = []

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'PDB_id']

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    resolution = get_resolution(pdb_id)
    resolution_list.append(resolution)

data_df['resolution'] = resolution_list
data_df.to_csv("/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v2_with_resolution.csv", index=False)

  1%|          | 1/183 [00:01<04:36,  1.52s/it]

Resolution for PDB ID 7E39: 3.7 Å


  1%|          | 2/183 [00:02<03:29,  1.16s/it]

Resolution for PDB ID 7E86: 2.9 Å


  2%|▏         | 3/183 [00:04<04:35,  1.53s/it]

...

Resolution for PDB ID 7S4S: 2.05 Å


 99%|█████████▉| 182/183 [07:21<00:00,  1.01it/s]

Resolution for PDB ID 7SN2: 4.3 Å


100%|██████████| 183/183 [07:21<00:00,  2.41s/it]

Resolution for PDB ID 7VMU: 2.89 Å


#### 3.2 标注链信息，打印辅助信息+手动标注+检查 v2->v3

In [ ]:
# 添加partner信息：手动，但是可以打印出每一个蛋白的链+序列，方便标注
# 得到v3文件，包括手动确认partner信息，且拆分多个抗体-抗原复合物，若相同抗体则保留分辨率较高的，而且删除了6M0J

ref_S1_seq = "TNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVC"

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v2_with_resolution.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs_origin"

for idx in range(len(data_df)):
    pdb_id = data_df.loc[idx, 'PDB_id']

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    chain_residue_mappint = {}

    for chain in pdb_structure:
        chain_id = chain.get_id()
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        aa_sequence = []
        
        for res in residues:
            res_id = int(res.get_id()[1])
            resname = res.get_resname()
            if not is_aa(resname):
                if resname != 'HOH':
                    logging.warning(f'PDB {pdb_id} Chain {chain.get_id()} Res {resname} not AA')
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]
            aa_sequence.append(token)

        seq = ''.join(aa_sequence)

        chain_residue_mappint[chain_id] = seq

    print(f"PDB ID: {pdb_id}")
    for chain_id, seq in chain_residue_mappint.items():
        print(f"Chain {chain_id}: {seq}")
    print()

In [ ]:
# 检查是否都去重完了
data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v3_with_resolution_hand_correct.csv"
data_df = pd.read_csv(data_path)
print(len(data_df))
# drop duplicates based on antibody_chain and partner, keep the one with the highest resolution
data_df = data_df.drop_duplicates(subset=['Antibody_name'], keep='first')
print(len(data_df))

#### 3.3 根据链信息，创造新的结构 v3->v4

In [ ]:
# 根据链信息提取结构，并判断标注的H和L是否都为轻链+重链，整理出一个v4文件

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v3_with_resolution_hand_correct.csv"
data_df = pd.read_csv(data_path)

pdb_src_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs_origin/"
pdb_target_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

data_save_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"

class ChainSelector_SARS(Select):
    def __init__(self, heavy_chain=None, light_chain=None, antigen_chains=None):
        self.heavy_chain = heavy_chain
        self.light_chain = light_chain
        self.antigen_chains = antigen_chains or []

    def accept_chain(self, chain):
        chain_id = chain.get_id()
        return (chain_id == self.heavy_chain or chain_id == self.light_chain or
                chain_id in self.antigen_chains)

pdb_id_list =  []
Antibody_name_list = []

h_chain_list = []
l_chain_list = []
antigen_chain_list = []

seq_list = []
cdr1_list = []
cdr2_list = []
cdr3_list = []

resolution_list = []

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'PDB_id']

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    heavy_chain = data_df.loc[idx, 'H_chain']
    light_chain = data_df.loc[idx, 'L_chain']
    antigen_chain = data_df.loc[idx, 'Antigen_chain']

    if not pd.notna(light_chain):
        light_chain = ''

    pdb_path = os.path.join(pdb_src_path, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    chain_ids = [chain.get_id() for chain in pdb_structure]

    # 检查h和lchain是否正确
    raw_H_seq = parse_biopython_structure(pdb_id, pdb_structure[heavy_chain])[heavy_chain]    

    truncked_H_seq = parse_biopython_structure(pdb_id, pdb_structure[heavy_chain], max_seq_len=113)[heavy_chain]
    chain_heavy_truncked = Chain(truncked_H_seq, scheme='chothia')
    assert chain_heavy_truncked.is_heavy_chain() , f"Heavy chain type mismatch in {pdb_id}"

    cdr1 = chain_heavy_truncked.cdr1_seq
    cdr2 = chain_heavy_truncked.cdr2_seq
    cdr3 = chain_heavy_truncked.cdr3_seq

    if not light_chain == '':
        raw_L_seq = parse_biopython_structure(pdb_id, pdb_structure[light_chain])[light_chain]
        chain_light = Chain(raw_L_seq, scheme='chothia')
        assert chain_light.is_light_chain(), f"Light chain type mismatch in {pdb_id}"

    # 存储对应链到新结构
    selector = ChainSelector_SARS(
        heavy_chain=heavy_chain,
        light_chain=light_chain,
        antigen_chains=[antigen_chain]
    )
    io = PDBIO()
    io.set_structure(structure)

    pdb_id_new = f"{pdb_id}_{heavy_chain}_{light_chain}_{antigen_chain}"

    output_pdb_file = os.path.join(pdb_target_path, f"{pdb_id_new}.pdb")
    io.save(output_pdb_file, select=selector)

    pdb_id_list.append(pdb_id_new)
    Antibody_name_list.append(data_df.loc[idx, 'Antibody_name'])
    h_chain_list.append(heavy_chain)
    l_chain_list.append(light_chain)
    antigen_chain_list.append(antigen_chain)

    seq_list.append(raw_H_seq)
    cdr1_list.append(cdr1)
    cdr2_list.append(cdr2)
    cdr3_list.append(cdr3)

    resolution_list.append(data_df.loc[idx, 'resolution'])

data = {
    'pdb_id': pdb_id_list,
    'Antibody_name': Antibody_name_list,
    'H_chain': h_chain_list,
    'L_chain': l_chain_list,
    'Antigen_chain': antigen_chain_list,
    'seq': seq_list,
    'cdr1': cdr1_list,
    'cdr2': cdr2_list,
    'cdr3': cdr3_list,
    'resolution': resolution_list
}

df = pd.DataFrame(data)
df.to_csv(data_save_path, index=False)

#### 3.4. 检查抗原是否都能突变，踢掉结构损失的，重新标注标号偏离的抗原+抗体 v4->v5

In [ ]:
# 检查1：初始坐标和RBD开头的坐标是否看上去无误

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

for idx in tqdm(range(len(data_df))):
    antigen_chain = data_df.loc[idx, 'Antigen_chain']
    pdb_id = data_df.loc[idx, 'pdb_id']

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[antigen_chain])

    # 检查1：初始坐标和RBD开头的坐标是否看上去无误
    seq_frag = []
    for key, value in pos_aa_mapping.items():
        if key >= 333 and key <= 340:
            seq_frag.append(f"{value}")

    seq_frag = ''.join(seq_frag)
    print(f"PDB ID: {pdb_id}, Antigen Chain: {antigen_chain}, Sequence Length: {len(pos_aa_mapping)}")
    print(f"Initial Segment: {seq_frag}...")
    print(f"Initial index: {list(pos_aa_mapping.keys())[0]}")


  1%|          | 2/182 [00:00<00:14, 12.59it/s]

PDB ID: 6XC2_H_L_A, Antigen Chain: A, Sequence Length: 193
Initial Segment: NLCPFGE...
Initial index: 334
PDB ID: 6XC4_H_L_A, Antigen Chain: A, Sequence Length: 199
Initial Segment: NLCPFGE...
Initial index: 334
PDB ID: 6XCM_H_L_C, Antigen Chain: C, Sequence Length: 985
Initial Segment: TNLCPFGE...
Initial index: 27


  3%|▎         | 6/182 [00:00<00:20,  8.75it/s]

PDB ID: 6XDG_B_D_E, Antigen Chain: E, Sequence Length: 194
Initial Segment: TNLCPFGE...
Initial index: 333
PDB ID: 6XDG_C_A_E, Antigen Chain: E, Sequence Length: 194
Initial Segment: TNLCPFGE...
Initial index: 333
PDB ID: 6XE1_H_L_E, Antigen Chain: E, Sequence Length: 207
Initial Segment: TNLCPFGE...
Initial index: 326


  4%|▍         | 8/182 [00:01<00:25,  6.86it/s]

...

PDB ID: 7RR0_B_C_A, Antigen Chain: A, Sequence Length: 194
Initial Segment: NLCPFGE...
Initial index: 334
PDB ID: 7S0B_A_B_F, Antigen Chain: F, Sequence Length: 198
Initial Segment: TNLCPFGE...
Initial index: 331


 99%|█████████▉| 180/182 [00:24<00:00,  9.22it/s]

PDB ID: 7S4S_H_L_A, Antigen Chain: A, Sequence Length: 193
Initial Segment: NLCPFGE...
Initial index: 334
PDB ID: 7SN2_H_L_A, Antigen Chain: A, Sequence Length: 223
Initial Segment: TNLCPFGE...
Initial index: 319


100%|██████████| 182/182 [00:25<00:00,  7.28it/s]

PDB ID: 7VMU_A__B, Antigen Chain: B, Sequence Length: 176
Initial Segment: LCPFGE...
Initial index: 335


In [9]:
# 对于检查1发现的有问题的，修正抗原链编号（就一个，难绷）

SPECIAL_ANTIGEN_MAPPING = {
    '7B27_C__A': 12,
}

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

for idx in range(len(data_df)):
    pdb_id = data_df.loc[idx, 'pdb_id']

    if pdb_id in SPECIAL_ANTIGEN_MAPPING:
        antigen_chain = data_df.loc[idx, 'Antigen_chain']
        index_bias = SPECIAL_ANTIGEN_MAPPING[pdb_id]

        pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
        pdb_save_path = pdb_path.replace('.pdb', '_renum.pdb')
        
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("pdb_id", pdb_path)
        pdb_structure = structure[0]

        antigen_chain = pdb_structure[antigen_chain]

        chain_res_id_list = []
        # ========== 第一阶段：记录原始res id，且使用临时ID避免冲突 ==========
        residues_in_chain = list(antigen_chain.get_residues())
        for residue in residues_in_chain:
            old_id = residue.get_id()
            het_flag, old_seq_id, old_icode = old_id
            chain_res_id_list.append(old_seq_id + index_bias)
            temp_id = (het_flag, 10000 + old_seq_id, old_icode)
            residue.id = temp_id
        
        # ========== 第二阶段：修改为最终编号 ==========
        for i, residue in enumerate(residues_in_chain):
            het_flag, temp_seq_id, old_icode = residue.get_id()
            new_seq_id = chain_res_id_list[i]
            new_id = (het_flag, new_seq_id, old_icode)
            residue.id = new_id
            
        io = PDBIO()
        io.set_structure(structure)
        io.save(pdb_save_path)
        print(f"Renumbered antigen chain {antigen_chain.get_id()} in {pdb_id} and saved to {pdb_save_path}")

# 手动检查后再把renum改回去原文件名

for idx in range(len(data_df)):
    pdb_id = data_df.loc[idx, 'pdb_id']

    if pdb_id in SPECIAL_ANTIGEN_MAPPING:
        antigen_chain = data_df.loc[idx, 'Antigen_chain']
        pdb_id = data_df.loc[idx, 'pdb_id']

        pdb_path = os.path.join(pdb_dir, f"{pdb_id}_renum.pdb")
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[antigen_chain])

        # 检查1：初始坐标和RBD开头的坐标是否看上去无误
        seq_frag = []
        for key, value in pos_aa_mapping.items():
            if key >= 333 and key <= 340:
                seq_frag.append(f"{value}")

        seq_frag = ''.join(seq_frag)
        print(f"PDB ID: {pdb_id}, Antigen Chain: {antigen_chain}, Sequence Length: {len(pos_aa_mapping)}")
        print(f"Initial Segment: {seq_frag}...")
        print(f"Initial index: {list(pos_aa_mapping.keys())[0]}")

Renumbered antigen chain A in 7B27_C__A and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7B27_C__A_renum.pdb
PDB ID: 7B27_C__A, Antigen Chain: A, Sequence Length: 202
Initial Segment: LCPFGE...
Initial index: 335


In [ ]:
# 检查2：突变位点的野生型氨基酸是否和预期一致

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

mutate_type_pos_mapping = {
    'gamma': 'K417N_E484K_N501Y',
    'delta': 'L452R_T478K',
    'omicron': 'G339D_S371L_S373P_S375F_K417N_N440K_G446S_S477N_T478K_E484A_Q493R_G496S_Q498R_N501Y_Y505H'
}

ANTIGEN_SPECIAL_PDB_IDS = []

for idx in tqdm(range(len(data_df))):
    antigen_chain = data_df.loc[idx, 'Antigen_chain']
    pdb_id = data_df.loc[idx, 'pdb_id']

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[antigen_chain])

    # 检查2：突变位点的野生型氨基酸是否和预期一致
    for mutate_type, mutate_infos in mutate_type_pos_mapping.items():
        for mutate_info in mutate_infos.split('_'):
            pos = mutate_info[1: -1]
            wt_aa = mutate_info[0]

            if pos_aa_mapping.get(int(pos), '') != wt_aa:
                print(f"Mutation position mismatch in {pdb_id} for {mutate_type} mutation at position {pos}: expected {wt_aa}, found {pos_aa_mapping.get(int(pos), '')}")
                ANTIGEN_SPECIAL_PDB_IDS.append(pdb_id)

print("Special PDB IDs with mutation mismatches:", set(ANTIGEN_SPECIAL_PDB_IDS))


  0%|          | 0/182 [00:00<?, ?it/s]

Mutation position mismatch in 6XC2_H_L_A for omicron mutation at position 446: expected G, found 


  7%|▋         | 12/182 [00:01<00:15, 10.96it/s]

Mutation position mismatch in 6XKQ_H_L_A for omicron mutation at position 371: expected S, found 
Mutation position mismatch in 6XKQ_H_L_A for omicron mutation at position 373: expected S, found 


  9%|▉         | 16/182 [00:01<00:18,  8.90it/s]

...

Mutation position mismatch in 7K9K_H_L_A for gamma mutation at position 484: expected E, found 
Mutation position mismatch in 7K9K_H_L_A for delta mutation at position 478: expected T, found 
Mutation position mismatch in 7K9K_H_L_A for omicron mutation at position 477: expected S, found 
Mutation position mismatch in 7K9K_H_L_A for omicron mutation at position 478: expected T, found 
Mutation position mismatch in 7K9K_H_L_A for omicron mutation at position 484: expected E, found 


 81%|████████  | 147/182 [00:17<00:02, 14.21it/s]

Mutation position mismatch in 7MZG_H_L_A for omicron mutation at position 446: expected G, found 


100%|██████████| 182/182 [00:20<00:00,  8.71it/s]

Special PDB IDs with mutation mismatches: {'6XKQ_H_L_A', '7E86_A_B_C', '7F62_H_L_A', '7K8T_H_L_C', '7MZG_H_L_A', '7CHP_H_L_E', '7K8Z_H_L_A', '7JV6_H_L_A', '7F63_H_L_A', '7K8V_H_L_A', '6XC2_H_L_A', '7K9K_H_L_A', '7A29_D__A', '7JVA_H_L_A'}


In [ ]:
# 14个有问题的，打开文件检查了下，确实缺东西，救不了
ANTIGEN_SPECIAL_PDB_IDS = ['6XKQ_H_L_A', '7E86_A_B_C', '7F62_H_L_A', '7K8T_H_L_C', '7MZG_H_L_A', '7CHP_H_L_E', '7K8Z_H_L_A',
                   '7JV6_H_L_A', '7F63_H_L_A', '7K8V_H_L_A', '6XC2_H_L_A', '7K9K_H_L_A', '7A29_D__A', '7JVA_H_L_A']
print(len(ANTIGEN_SPECIAL_PDB_IDS))

14


In [ ]:
# 检查3：抗体编号是否从1开始

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

ANTIBODY_SPECIAL_PDB_IDS = []

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'pdb_id']
    h_chain = data_df.loc[idx, 'H_chain']
    l_chain = data_df.loc[idx, 'L_chain']

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    h_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[h_chain])
    # print(f"H Initial index: {list(h_chain_mapping.keys())[0]}")
    if list(h_chain_mapping.keys())[0] != 1:
        print(f"Heavy chain initial index not 1 in {pdb_id}")
        ANTIBODY_SPECIAL_PDB_IDS.append(pdb_id)

    if not pd.notna(l_chain):
        continue

    l_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[l_chain])
    # print(f"L Initial index: {list(l_chain_mapping.keys())[0]}")
    if list(h_chain_mapping.keys())[0] != 1:
        print(f"Light chain initial index not 1 in {pdb_id}")
        ANTIBODY_SPECIAL_PDB_IDS.append(pdb_id)
        
print("Special PDB IDs with antibody chain numbering issues:", set(ANTIBODY_SPECIAL_PDB_IDS))

In [18]:
# 对于发现的非从1开始编号的抗体，重编号
ANTIBODY_SPECIAL_PDB_IDS = ['7VMU_A__B', '7MEJ_B__R', '7R8L_H_L_E', '7MLZ_H_L_A', '7BZ5_H_L_A', '7CWM_H_L_A', '7K8T_H_L_C', '7MMO_A_B_C', '7JV6_H_L_A', '7JMP_H_L_A', '6XCM_H_L_C', '7DX4_H_L_E', '7N4M_H_L_A', '7K90_H_L_B', '7D0D_H_L_A', '7E86_A_B_C', '7L5B_H_L_A', '7K9Z_B_A_E', '7MZL_H_L_A', '7K43_H_L_A', '7E7Y_A_B_R', '7E3C_B_C_A', '7MDW_B__R', '7FG3_D__A', '7CDI_H_L_E', '6ZCZ_F__E', '6XE1_H_L_E', '7M7W_A_B_S', '7D0B_H_L_A', '7CWN_E_D_B', '7M3I_A_B_C', '7CHE_A_B_R', '7BYR_H_L_B', '7FG2_D__A', '7JX3_H_L_R', '7KN7_B__A', '7KN5_E__A']
print(len(ANTIBODY_SPECIAL_PDB_IDS))

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'pdb_id']

    if pdb_id not in ANTIBODY_SPECIAL_PDB_IDS:
        continue

    h_chain = data_df.loc[idx, 'H_chain']
    l_chain = data_df.loc[idx, 'L_chain']

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    h_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[h_chain])
    if list(h_chain_mapping.keys())[0] != 1:
        
        h_chain_model = pdb_structure[h_chain]

        chain_res_id_list = []
        # ========== 第一阶段：记录原始res id，且使用临时ID避免冲突 ==========
        residues_in_chain = list(h_chain_model.get_residues())
        for new_idx, residue in enumerate(residues_in_chain, start=1):
            old_id = residue.get_id()
            het_flag, old_seq_id, old_icode = old_id
            chain_res_id_list.append(new_idx)
            temp_id = (het_flag, 10000 + old_seq_id, old_icode)
            residue.id = temp_id
        
        # ========== 第二阶段：修改为最终编号 ==========
        for i, residue in enumerate(residues_in_chain):
            het_flag, temp_seq_id, old_icode = residue.get_id()
            new_seq_id = chain_res_id_list[i]
            new_id = (het_flag, new_seq_id, old_icode)
            residue.id = new_id
            
    if pd.notna(l_chain):
        l_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[l_chain])
        if list(l_chain_mapping.keys())[0] != 1:
            
            l_chain_model = pdb_structure[l_chain]

            chain_res_id_list = []
            # ========== 第一阶段：记录原始res id，且使用临时ID避免冲突 ==========
            residues_in_chain = list(l_chain_model.get_residues())
            for new_idx, residue in enumerate(residues_in_chain, start=1):
                old_id = residue.get_id()
                het_flag, old_seq_id, old_icode = old_id
                chain_res_id_list.append(new_idx)
                temp_id = (het_flag, 10000 + old_seq_id, old_icode)
                residue.id = temp_id
            
            # ========== 第二阶段：修改为最终编号 ==========
            for i, residue in enumerate(residues_in_chain):
                het_flag, temp_seq_id, old_icode = residue.get_id()
                new_seq_id = chain_res_id_list[i]
                new_id = (het_flag, new_seq_id, old_icode)
                residue.id = new_id

    # move old pdb file to backup
    pdb_old_path = pdb_path.replace('.pdb', '_old.pdb')
    os.rename(pdb_path, pdb_old_path)

    io = PDBIO()
    io.set_structure(structure)
    io.save(pdb_path)
    print(f"Renumbered antibody chains in {pdb_id} and saved to {pdb_path}")

37


  0%|          | 0/182 [00:00<?, ?it/s]

  2%|▏         | 3/182 [00:00<00:45,  3.93it/s]

Renumbered antibody chains in 6XCM_H_L_C and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/6XCM_H_L_C.pdb


  7%|▋         | 12/182 [00:01<00:13, 12.21it/s]

...

Renumbered antibody chains in 7MLZ_H_L_A and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7MLZ_H_L_A.pdb
Renumbered antibody chains in 7MMO_A_B_C and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7MMO_A_B_C.pdb


 87%|████████▋ | 158/182 [00:09<00:01, 22.60it/s]

Renumbered antibody chains in 7MZL_H_L_A and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7MZL_H_L_A.pdb
Renumbered antibody chains in 7N4M_H_L_A and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7N4M_H_L_A.pdb


100%|██████████| 182/182 [00:10<00:00, 17.98it/s]

Renumbered antibody chains in 7R8L_H_L_E and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7R8L_H_L_E.pdb
Renumbered antibody chains in 7VMU_A__B and saved to /home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/7VMU_A__B.pdb


In [ ]:
# 修正完后再check一遍
data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

ANTIBODY_SPECIAL_PDB_IDS_new = []

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'pdb_id']
    h_chain = data_df.loc[idx, 'H_chain']
    l_chain = data_df.loc[idx, 'L_chain']    

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    h_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[h_chain])
    # print(f"H Initial index: {list(h_chain_mapping.keys())[0]}")
    if list(h_chain_mapping.keys())[0] != 1:
        print(f"Heavy chain initial index not 1 in {pdb_id}")
        ANTIBODY_SPECIAL_PDB_IDS_new.append(pdb_id)

    if not pd.notna(l_chain):
        continue

    l_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[l_chain])
    # print(f"L Initial index: {list(l_chain_mapping.keys())[0]}")
    if list(h_chain_mapping.keys())[0] != 1:
        print(f"Light chain initial index not 1 in {pdb_id}")
        ANTIBODY_SPECIAL_PDB_IDS_new.append(pdb_id)
        
print("Special PDB IDs with antibody chain numbering issues:", set(ANTIBODY_SPECIAL_PDB_IDS_new))

  3%|▎         | 6/182 [00:01<00:38,  4.58it/s]

Heavy chain initial index not 1 in 6XE1_H_L_E
Light chain initial index not 1 in 6XE1_H_L_E


 73%|███████▎  | 133/182 [00:22<00:06,  7.41it/s]

Heavy chain initial index not 1 in 7M3I_A_B_C
Light chain initial index not 1 in 7M3I_A_B_C


 79%|███████▉  | 144/182 [00:23<00:03, 10.37it/s]

Heavy chain initial index not 1 in 7MMO_A_B_C
Light chain initial index not 1 in 7MMO_A_B_C


100%|██████████| 182/182 [00:28<00:00,  6.48it/s]

Special PDB IDs with antibody chain numbering issues: {'7MMO_A_B_C', '6XE1_H_L_E', '7M3I_A_B_C'}


In [ ]:
# 剩下三个没法解决，记录偏置吧。。。
ANTIBODY_SPECIAL_PDB_IDS_mapping = {'7MMO_A_B_C': 1, '6XE1_H_L_E': 1, '7M3I_A_B_C': 1}

In [4]:
# 筛选出新的csv
ANTIGEN_SPECIAL_PDB_IDS = ['6XKQ_H_L_A', '7E86_A_B_C', '7F62_H_L_A', '7K8T_H_L_C', '7MZG_H_L_A', '7CHP_H_L_E', '7K8Z_H_L_A',
                   '7JV6_H_L_A', '7F63_H_L_A', '7K8V_H_L_A', '6XC2_H_L_A', '7K9K_H_L_A', '7A29_D__A', '7JVA_H_L_A']
ANTIBODY_SPECIAL_PDB_IDS_mapping = {'7MMO_A_B_C': 1, '6XE1_H_L_E': 1, '7M3I_A_B_C': 1}

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v4_with_resolution_hand_correct_seperate_pdb.csv"
data_df = pd.read_csv(data_path)

h_chain_bias = []
antigen_flag = []

for idx in range(len(data_df)):
    pdb_id = data_df.loc[idx, 'pdb_id']
    if pdb_id in ANTIGEN_SPECIAL_PDB_IDS:
        antigen_flag.append(0)
    else:
        antigen_flag.append(1)
    
    if pdb_id in ANTIBODY_SPECIAL_PDB_IDS_mapping.keys():
        h_chain_bias.append(ANTIBODY_SPECIAL_PDB_IDS_mapping[pdb_id])
    else:
        h_chain_bias.append(0)

data_df['chain_bias'] = h_chain_bias
data_df['antigen_flag'] = antigen_flag
data_df = data_df[data_df['antigen_flag'] == 1]

save_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v5_with_resolution_hand_correct_seperate_pdb_correct_start_idx.csv"
data_df.to_csv(save_path, index=False)

In [ ]:
# 检查4：抗体序列是否连续，只需要查h链（不对，仔细一想，就算断了，自己重新编码后就没有任何问题了）
ANTIBODY_SPECIAL_PDB_IDS = ['7VMU_A__B', '7MEJ_B__R', '7R8L_H_L_E', '7MLZ_H_L_A', '7BZ5_H_L_A', '7CWM_H_L_A', '7K8T_H_L_C',
                            '7MMO_A_B_C', '7JV6_H_L_A', '7JMP_H_L_A', '6XCM_H_L_C', '7DX4_H_L_E', '7N4M_H_L_A', '7K90_H_L_B',
                            '7D0D_H_L_A', '7E86_A_B_C', '7L5B_H_L_A', '7K9Z_B_A_E', '7MZL_H_L_A', '7K43_H_L_A', '7E7Y_A_B_R',
                            '7E3C_B_C_A', '7MDW_B__R', '7FG3_D__A', '7CDI_H_L_E', '6ZCZ_F__E', '6XE1_H_L_E', '7M7W_A_B_S',
                            '7D0B_H_L_A', '7CWN_E_D_B', '7M3I_A_B_C', '7CHE_A_B_R', '7BYR_H_L_B', '7FG2_D__A', '7JX3_H_L_R',
                            '7KN7_B__A', '7KN5_E__A']
print(len(ANTIBODY_SPECIAL_PDB_IDS))

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v5_with_resolution_hand_correct_seperate_pdb_correct_start_idx.csv"
data_df = pd.read_csv(data_path)

pdb_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"

for idx in tqdm(range(len(data_df))):
    pdb_id = data_df.loc[idx, 'pdb_id']
    h_chain = data_df.loc[idx, 'H_chain']
    l_chain = data_df.loc[idx, 'L_chain']    

    pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
    if pdb_id in ANTIBODY_SPECIAL_PDB_IDS:
        pdb_path.replace('.pdb', '_old.pdb')  # 得去看原始的是否连续，重新编完的肯定连续了

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    h_chain_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[h_chain])
    # 检查4：抗体序列是否连续，只需要查h链
    h_chain_residue_ids = list(h_chain_mapping.keys())
    expected_ids = list(range(h_chain_residue_ids[0], h_chain_residue_ids[0] + len(h_chain_residue_ids)))
    if h_chain_residue_ids != expected_ids:
        print(f"Heavy chain residues not continuous in {pdb_id}:")
        print(f"Actual IDs: {h_chain_residue_ids}")
        print(f"Expect IDs: {expected_ids}")
    

#### 3.5 突变RBD

In [ ]:
# 创建一个新的csv？不用吧，在突变代码里面写就行了，突变pdb文件名记得改成gamma这种。这里只检查是否都完成了突变

import pandas as pd
import os
import sys

data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v5_with_resolution_hand_correct_seperate_pdb_correct_start_idx.csv"
data_df = pd.read_csv(data_path)

wt_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs/"
fix_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs_fixed/"
mut_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs_mutated/"

df = pd.read_csv(data_path)
for idx in range(len(df)):
    row = df.iloc[idx]
    pdb_id = row['pdb_id']

    wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
    fix_pdb_file = os.path.join(fix_dir, f"{pdb_id}.pdb")
    
    if not os.path.exists(fix_pdb_file):
        raise FileNotFoundError(f"Fixed PDB file {fix_pdb_file} does not exist")
    
    for mutate_info_new in ['gamma', 'delta', 'omicron']:
        mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

        if not os.path.exists(mut_pdb_file):
            print(f"Mutated PDB file {mut_pdb_file} does not exist")

In [7]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, chain_id, mut_dir = args

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    # 3. update mutate_info
    for mutate_type, mutant in mutate_type_pos_mapping.items():

        pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_type}.pdb")

        mutate_infos = []
        for mutate_info in mutant.split('_'):
            mutate_infos.append(f"{mutate_info[0]}{chain_id}{mutate_info[1:-1]}{mutate_info[-1]}")
        mutate_infos = ','.join(mutate_infos)
        # 3. end

        try:
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure(pdb_id, pdb_path)
            pdb_structure = structure[0]

            for mutate_info in mutate_infos.split(','):
                chain_id = mutate_info[1]

                pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

                # Check mutate info
                if pos_aa_mapping.get(int(mutate_info[2:-1]), '') != mutate_info[-1]:
                    error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutate_info}: expected {mutate_info[-1]}, found {pos_aa_mapping.get(int(mutate_info[2:-1]), '')}, all mutations are {mutate_infos}"
                    return error_msg
        except Exception as e:
            error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
            return error_msg

    return None

# 1. read data and set path
data_path = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/sars_evo_v5_with_resolution_hand_correct_seperate_pdb_correct_start_idx.csv"
data_df = pd.read_csv(data_path)

mut_dir = "/home/lfj/projects_dir/MERF/data/SARS_COV_2/PDBs_mutated/"
# 1. end

mutate_type_pos_mapping = {
    'gamma': 'K417N_E484K_N501Y',
    'delta': 'L452R_T478K',
    'omicron': 'G339D_S371L_S373P_S375F_K417N_N440K_G446S_S477N_T478K_E484A_Q493R_G496S_Q498R_N501Y_Y505H'
}

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'Antigen_chain'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 168/168 [00:04<00:00, 38.07it/s]


All 168 mutations are correct!


### 4. 评估指标 - Rosetta energy

In [ ]:
# 见scripts.calculate_ddG_pyrosetta.calculate_ddG 这里import有些问题

wt_path = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs_fixed/7FAE.pdb"
mut_path = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs_mutated/7FAE_AH53G.pdb"
partner = "HL_A"
output_dir = "/home/lfj/projects_dir/MERF/scripts/docking_test/"

# results = calculate_ddG(wt_path, mut_path, partner, output_dir)

### 5. 评估指标 - ipSAE

In [ ]:
# 见scripts.calculate_ipsae